# Validate Resource Isolation

**Goal**: prove that the Spark driver runs inside the **Livy container** (the simulated edge node),
not inside *your* JupyterLab container.

```
┌──────────────────────────────────────────────────────────────┐
│  YOUR BROWSER                                                │
│    └─► singleuser container  (JupyterLab + SparkMagic)      │
│              │  REST HTTP (POST /sessions, /statements)      │
│              ▼                                               │
│         Livy container  ◄─── THIS is the "edge node"        │
│           (Spark driver lives here)                          │
│              │  spark://spark-master:7077                    │
│              ▼                                               │
│         spark-master → spark-worker  (executors here)        │
└──────────────────────────────────────────────────────────────┘
```

Run each cell in order.  
While **Cell 8** (the heavy job) is running, switch to a second terminal and run:
```
docker stats --no-stream
```
You will see CPU/memory activity in **livy** and **spark-worker** but almost none in the singleuser container.

## Step 0 — SparkMagic session info

`%%info` shows the active Livy session — its ID, state, and the Livy endpoint that is serving it.

In [ ]:
%%info

## Step 1 — Where does the PySpark code actually execute?

This cell calls `socket.gethostname()` **inside the Livy interpreter** (i.e. inside the Livy JVM/Python worker, not in your JupyterLab container).

You will see the **Livy container's hostname** — not `jupyterhub-*` or your own container name.
That is your proof that the driver runs on the edge node, never in the user container.

In [ ]:
import socket
print("Code is executing inside container:", socket.gethostname())
print("Driver host seen by SparkContext   :", sc.master)

## Step 2 — Spark Application UI

The Spark driver registers a web UI inside the Spark master.  
`sc.uiWebUrl` returns the URL of _your_ application's detail page on the Spark master UI.

In [ ]:
print("Spark Master UI (all apps) : http://localhost:8080")
print("This session's app URL     :", sc.uiWebUrl)
print("SparkContext app name       :", sc.appName)
print("Spark version               :", sc.version)

## Step 3 — Executor allocation

List the executors that have registered with this SparkContext.  
Each executor runs inside the **spark-worker** container.
The `driver` executor entry represents the process inside Livy.

In [ ]:
# executorList() returns a Scala Seq -- convert to a Java List so Python can iterate it
executor_seq = sc._jvm.org.apache.spark.SparkContext.getOrCreate().statusStore().executorList(True)
executor_info = sc._jvm.scala.collection.JavaConverters.seqAsJavaList(executor_seq)

print(f"{'ID':<8} {'Host':<30} {'Cores':<8} {'Memory (MB)':<14} {'Active Tasks'}")
print("-" * 70)
for e in executor_info:
    print(f"{e.id():<8} {e.hostPort():<30} {e.totalCores():<8} {e.maxMemory() // (1024**2):<14} {e.activeTasks()}")

## Step 4 — Active Livy sessions (REST API)

From inside the singleuser container, call the Livy REST API directly.  
This is a `%%local` cell — it runs in the singleuser container (Python kernel, no Livy), so it shows
that SparkMagic is just an HTTP client.

In [ ]:
%%local
import urllib.request, json

url = "http://livy:8998/sessions"
with urllib.request.urlopen(url) as resp:
    data = json.loads(resp.read())

print(f"Active Livy sessions: {data['total']}")
for s in data["sessions"]:
    print(f"  id={s['id']}  state={s['state']}  kind={s['kind']}  appId={s.get('appId','—')}")

## Step 5 — Multi-user isolation test

If two users (alice and bob) are logged in simultaneously, each has their own Livy session,
their own SparkContext, and their own set of executors — completely isolated.

Re-run the `%%local` cell above after a second user logs in and you'll see **two** sessions listed.

## Step 6 — Heavy computation (watch docker stats while this runs)

While this cell is running, open a terminal on the host and run:
```powershell
docker stats --no-stream
```

**Expected observations:**
| Container | CPU / Memory | Why |
|---|---|---|
| `learn-jupyterhub-with-livy-spark-worker-1` | **HIGH** | Executors run here |
| `learn-jupyterhub-with-livy-livy-1` | **MEDIUM** | Spark driver lives here |
| your singleuser container (`jupyter-alice`) | **~0%** | Just an HTTP client — waits for result |
| `learn-jupyterhub-with-livy-jupyterhub-1` | **~0%** | Hub is idle |

This is the resource isolation guarantee: the user's container never consumes cluster resources.

In [ ]:
# CPU-intensive: compute pi with Monte-Carlo (runs on executor)
import random

NUM_SAMPLES = 5_000_000

def inside_unit_circle(_):
    x, y = random.random(), random.random()
    return x*x + y*y < 1.0

count = sc.parallelize(range(NUM_SAMPLES), 4).filter(inside_unit_circle).count()
pi_estimate = 4.0 * count / NUM_SAMPLES
print(f"pi ≈ {pi_estimate:.6f}  (expected ~3.141593)")

## Step 7 — Resource governance: per-user caps vs executor sizing

There are **two independent resource controls** in this stack:

| Control | Where configured | What it governs |
|---|---|---|
| `c.DockerSpawner.mem_limit = "2G"` | `jupyterhub_config.py` | Memory of the **singleuser container** (the thin client) |
| `executorMemory: "1G"` | `config/sparkmagic/config.json` | Memory of each **Spark executor** on the worker |
| `SPARK_WORKER_MEMORY: "2G"` | `docker-compose.yml` | Total memory available to all executors on a worker |

In production (YARN), `executorMemory` maps to a YARN queue allocation.  
The singleuser container stays small regardless of the job size.

### Horizontal scaling
Add a second worker at runtime — Spark will automatically use it for new tasks:
```powershell
# from the host (not this notebook)
docker compose up --scale spark-worker=3 -d
```
Re-run Step 3 after scaling and you will see additional executor entries.